# Notebook 08 — EKF Consistency: NIS and NEES

**Learning objectives**
- Understand what consistency means for a state estimator.
- Define and compute NIS and NEES.
- Use chi-square thresholds for practical checks.
- Detect mis-specified measurement noise `R` using NIS.

**Prerequisites**
- Notebook 06–07.


## Table of Contents

- [1) Intuition: what is consistency?](#1-intuition-what-is-consistency)
- [2) Monte-Carlo simulation setup](#2-monte-carlo-simulation-setup)
- [3) Mis-specified `R`: detect with NIS](#3-mis-specified-r-detect-with-nis)
- [Exercises](#exercises)

---

## Navigation

- Previous: [ntbk-07-ekf-2d-tracking-example.ipynb](./ntbk-07-ekf-2d-tracking-example.ipynb)
- Next: [ntbk-09-differential-drive-kinematics.ipynb](./ntbk-09-differential-drive-kinematics.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/ekf_slam.py`
- `tests/test_robust_ekf_suite.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 107
set_seed(SEED)
plt.rcdefaults()


## 1) Intuition: what is consistency?

A filter is **consistent** if its predicted uncertainty matches actual errors statistically.

- If the filter is overconfident, real errors are larger than covariance suggests.
- If underconfident, covariance is too large and the filter is conservative.

Two common tests:

1. **NIS (Normalized Innovation Squared)**
\[
\mathrm{NIS}_k = y_k^T S_k^{-1} y_k
\]
where $y_k$ is innovation and $S_k$ is innovation covariance.

2. **NEES (Normalized Estimation Error Squared)**
\[
\mathrm{NEES}_k = e_k^T P_k^{-1} e_k,\quad e_k = x_k-\hat{x}_k
\]

In words:
- NIS checks measurement-space consistency.
- NEES checks state-space consistency (needs ground truth).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)


In [ ]:
# Approximate chi-square 95% interval endpoints for common degrees of freedom.
# (Hardcoded to keep notebook dependency-light: numpy + matplotlib only.)
CHI2_95_INTERVAL = {
    1: (0.001, 5.024),
    2: (0.051, 7.378),
    3: (0.216, 9.348),
    4: (0.484, 11.143),
}


## 2) Monte-Carlo simulation setup

We use a simple 2D position tracker:

\[
x_k = x_{k-1} + \Delta t\,u_k + w_k,\qquad z_k = x_k + v_k
\]

The filter assumes `Q_assumed` and `R_assumed`. The world uses `Q_true` and `R_true`.
When assumptions match truth, NIS/NEES statistics should fall in expected ranges often.


In [ ]:
def run_one_trial(N=60, dt=0.1, Q_true=None, R_true=None, Q_assumed=None, R_assumed=None):
    if Q_true is None:
        Q_true = np.diag([0.01, 0.01])
    if R_true is None:
        R_true = np.diag([0.2**2, 0.2**2])
    if Q_assumed is None:
        Q_assumed = Q_true.copy()
    if R_assumed is None:
        R_assumed = R_true.copy()

    x_true = np.zeros(2)
    x_hat = np.array([0.5, -0.3], dtype=float)
    P = np.diag([1.0, 1.0])

    nis_values, nees_values = [], []

    for k in range(N):
        t = k * dt
        u = np.array([1.0 + 0.2*np.cos(0.3*t), 0.4*np.sin(0.6*t)])

        # True process
        w = np.random.multivariate_normal(mean=np.zeros(2), cov=Q_true)
        x_true = x_true + dt * u + w

        # Predict
        x_hat = x_hat + dt * u
        P = P + Q_assumed

        # Measurement
        v = np.random.multivariate_normal(mean=np.zeros(2), cov=R_true)
        z = x_true + v

        # Update
        y = z - x_hat
        S = P + R_assumed
        K = P @ np.linalg.inv(S)

        x_hat = x_hat + K @ y
        P = (np.eye(2) - K) @ P

        nis = float(y.T @ np.linalg.inv(S) @ y)
        e = x_true - x_hat
        nees = float(e.T @ np.linalg.inv(P) @ e)
        nis_values.append(nis)
        nees_values.append(nees)

    return np.asarray(nis_values), np.asarray(nees_values)


In [ ]:
# Many runs to build empirical distribution.
M = 250
all_nis, all_nees = [], []
for _ in range(M):
    nis, nees = run_one_trial()
    all_nis.append(nis)
    all_nees.append(nees)

all_nis = np.concatenate(all_nis)
all_nees = np.concatenate(all_nees)
print('Mean NIS:', all_nis.mean())
print('Mean NEES:', all_nees.mean())


In [ ]:
dof = 2
lower, upper = CHI2_95_INTERVAL[dof]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_nis, bins=50, density=True, alpha=0.7, color='tab:blue')
axes[0].axvline(lower, color='k', linestyle='--', label='95% bounds')
axes[0].axvline(upper, color='k', linestyle='--')
axes[0].axvline(dof, color='tab:red', linestyle=':', label='expected mean=dof')
axes[0].set_title('NIS histogram')
axes[0].set_xlabel('NIS')
axes[0].legend()

axes[1].hist(all_nees, bins=50, density=True, alpha=0.7, color='tab:green')
axes[1].axvline(lower, color='k', linestyle='--', label='95% bounds')
axes[1].axvline(upper, color='k', linestyle='--')
axes[1].axvline(dof, color='tab:red', linestyle=':', label='expected mean=dof')
axes[1].set_title('NEES histogram')
axes[1].set_xlabel('NEES')
axes[1].legend()

plt.tight_layout()


## 3) Mis-specified `R`: detect with NIS

Now we intentionally make the filter **too optimistic** about measurement noise:

- true `R_true = diag(0.2^2, 0.2^2)`
- assumed `R_assumed = diag(0.05^2, 0.05^2)`

If we understate sensor noise, innovations look "too large" relative to `S`, so NIS inflates.


In [ ]:
all_nis_badR = []
for _ in range(M):
    nis_bad, _ = run_one_trial(
        R_true=np.diag([0.2**2, 0.2**2]),
        R_assumed=np.diag([0.05**2, 0.05**2]),
    )
    all_nis_badR.append(nis_bad)
all_nis_badR = np.concatenate(all_nis_badR)

plt.figure(figsize=(9,4))
plt.hist(all_nis, bins=50, density=True, alpha=0.5, label='well specified R')
plt.hist(all_nis_badR, bins=50, density=True, alpha=0.5, label='underestimated R')
plt.axvline(upper, color='k', linestyle='--', label='95% upper bound (dof=2)')
plt.axvline(dof, color='tab:red', linestyle=':', label='expected mean')
plt.title('NIS shifts right when R is underestimated')
plt.xlabel('NIS')
plt.legend()
plt.tight_layout()

print('Mean NIS (good R):', all_nis.mean())
print('Mean NIS (bad R): ', all_nis_badR.mean())


## Exercises

1. Overestimate `R_assumed` instead (e.g., `0.6^2`) and inspect NIS shift.
2. Keep `R` correct but underestimate `Q_assumed`. How does NEES respond?
3. Change confidence level from 95% to another level by replacing threshold values.

<details>
<summary><b>Optional solution hints</b></summary>

- Overestimated `R` usually shifts NIS left (filter becomes conservative in measurement space).
- Underestimated `Q` often causes NEES inflation during maneuvers.
- For production systems, run these checks online over sliding windows.

</details>


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
